# Notebook 06 — Topic modeling (BERTopic) with cross-embedding validation

**Project:** Study 2 — Chatbot Neuroticism (DGK Lab)  
**Pipeline stage:** 6 of 8 (NLP Colab notebooks) + 1 integration Rmd  
**Runtime:** ~30–45 minutes (4 embeddings × 4 questions × BERTopic fit)  
**API costs:** ~$2.40–3.20 (Opus 4.7 topic labeling)

## Purpose

Run **data-driven topic discovery** on the four primary OE responses as a supplementary, independent check on the rubric-based coding from Notebooks 03–05. Where the rubric is **deductive** (Opus authored the rubric themes from a 96-response sample, then three LLMs applied them), topic modeling is **inductive** in a different sense — embedding geometry + clustering surfaces structure without any rubric anchoring.

Three substantive questions Notebook 06 answers:

1. **Does the rubric have coverage gaps?** Are there topics that the BERTopic clusters surface but the rubric misses?
2. **Are the rubric themes data-grounded?** Do topics roughly correspond to rubric categories, providing convergent evidence?
3. **Does topic structure replicate across embedding models?** We run BERTopic on all four embeddings (mpnet, Voyage, OpenAI, Qwen3-8B) and compute cross-embedding ARI as a stability check.

## Approach

- **One BERTopic model per (oe_field × embedding)**: 4 questions × 4 embeddings = 16 models
- **Primary embedding: all-mpnet-base-v2** (BERTopic's reference embedding and our preregistered choice)
- **Robustness: Voyage-3-large, OpenAI text-embedding-3-large, Qwen3-Embedding-8B**
- **Topic labeling via Opus 4.7**: each topic gets a 3–5 word interpretable label from top c-TF-IDF terms + 3 exemplar responses (primary embedding only; ~40-50 calls)
- **Topic–rubric alignment**: for the primary embedding, compute co-occurrence of each topic with each rubric code (descriptive)
- **Cross-embedding agreement**: Adjusted Rand Index between embedding pairs, per question

## Important caveat (must appear in Methods)

BERTopic was developed and is generally validated on document collections N ≥ 500. Our per-question samples are N ≈ 118–128 substantive responses. We use topic modeling **descriptively, not confirmatorily**, to flag potential coverage gaps and provide convergent evidence on rubric structure. Topic counts and exact cluster boundaries should be interpreted as exploratory.

## Inputs

- `data/derived/open_ends_long.parquet` (response text, from Notebook 01)
- `data/derived/embeddings/all_mpnet_base_v2.parquet` (primary)
- `data/derived/embeddings/voyage_3_large.parquet`
- `data/derived/embeddings/openai_3_large.parquet`
- `data/derived/embeddings/qwen3_8b.parquet`
- `data/derived/coding/consensus_long.parquet` (for topic-rubric alignment)

## Outputs

- `data/derived/topics/topics_per_response.parquet` — primary mpnet topic assignments per response
- `data/derived/topics/topic_summary.csv` — per (oe_field × topic_id): label, top_terms, n_docs, exemplar, condition prevalence
- `data/derived/topics/topic_rubric_alignment.csv` — for each topic, which rubric codes are over-represented
- `data/derived/topics/cross_embedding_agreement.csv` — pairwise ARI by question
- `data/derived/topics/topics_other_embeddings/` — supplementary topic info from Voyage/OpenAI/Qwen3
- `data/derived/topics/topics_manifest.json`

## API keys required

- `ANTHROPIC_API_KEY` — for Opus 4.7 topic labeling


## 1. Environment setup

In [ ]:
import sys, os, json, time, hashlib
from pathlib import Path
from datetime import datetime
from typing import Dict, List, Optional, Tuple, Any

# Install BERTopic and deps if needed
try:
    from bertopic import BERTopic
except ImportError:
    print('Installing BERTopic and dependencies...')
    import subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                           'bertopic', 'umap-learn', 'hdbscan',
                           'sentence-transformers', 'scikit-learn'])
    from bertopic import BERTopic

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import adjusted_rand_score
from umap import UMAP
from hdbscan import HDBSCAN
!pip install anthropic
import anthropic

print(f'pandas: {pd.__version__}')
print(f'numpy:  {np.__version__}')
import bertopic
print(f'BERTopic: {bertopic.__version__}')


## 2. Project paths

In [ ]:
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    PROJECT_ROOT = Path('/content/drive/MyDrive/DGK Lab/Reasoning Lab/'
                        'DGK Lab - AI Training Data, CharacterAI, Tort Liability/'
                        'Analysis/Study 2')
else:
    PROJECT_ROOT = Path.cwd().parent if (Path.cwd().name == 'NLP') else Path.cwd()

DATA_DERIVED   = PROJECT_ROOT / 'data' / 'derived'
EMB_DIR        = DATA_DERIVED / 'embeddings'
CODING_DIR     = DATA_DERIVED / 'coding'
TOPICS_DIR     = DATA_DERIVED / 'topics'
TOPICS_OTHER   = TOPICS_DIR / 'topics_other_embeddings'

for d in [TOPICS_DIR, TOPICS_OTHER]:
    d.mkdir(parents=True, exist_ok=True)

PATH_OES       = DATA_DERIVED / 'open_ends_long.parquet'
PATH_MPNET     = EMB_DIR / 'all_mpnet_base_v2.parquet'
PATH_VOYAGE    = EMB_DIR / 'voyage_3_large.parquet'
PATH_OPENAI    = EMB_DIR / 'openai_3_large.parquet'
PATH_QWEN3     = EMB_DIR / 'qwen3_8b.parquet'
PATH_CONSENSUS = CODING_DIR / 'consensus_long.parquet'

required = [PATH_OES, PATH_MPNET, PATH_VOYAGE, PATH_OPENAI, PATH_QWEN3, PATH_CONSENSUS]
for p in required:
    if not p.exists():
        raise FileNotFoundError(f'Missing input: {p}')

print(f'Project root: {PROJECT_ROOT}')
print(f'Topics dir:   {TOPICS_DIR}')


## 3. Configuration

In [ ]:
# Embedding models to run (primary first)
EMBEDDING_MODELS = {
    'mpnet':   {'path': PATH_MPNET,   'is_primary': True,  'dim': 768,  'label': 'all-mpnet-base-v2'},
    'voyage':  {'path': PATH_VOYAGE,  'is_primary': False, 'dim': 1024, 'label': 'voyage-3-large'},
    'openai':  {'path': PATH_OPENAI,  'is_primary': False, 'dim': 3072, 'label': 'text-embedding-3-large'},
    'qwen3':   {'path': PATH_QWEN3,   'is_primary': False, 'dim': 4096, 'label': 'qwen3-embedding-8b'},
}
PRIMARY_KEY = 'mpnet'

# Questions
TARGET_QUESTIONS = ['oe_experience', 'oe_person_or_tool', 'oe_responsibility', 'oe_compensation']

# Filter for which responses get topic-modeled — match Notebook 04's coding filter
MIN_WORDS_FOR_TOPICS = 3

# BERTopic hyperparameters — tuned for our small per-question N (~120)
# Sticking close to defaults but reducing minimum cluster size since our corpus is small
UMAP_PARAMS = {
    'n_neighbors':  10,   # default 15; reduced for small corpus
    'n_components': 5,    # default 5
    'min_dist':     0.0,  # default 0.0 (encourages tight clusters)
    'metric':       'cosine',  # appropriate for normalized embeddings
    'random_state': 42,   # reproducibility
}

HDBSCAN_PARAMS = {
    'min_cluster_size': 8,    # default 10; reduced for small corpus
    'min_samples':      None, # default = min_cluster_size; let HDBSCAN decide
    'metric':           'euclidean',
    'cluster_selection_method': 'eom',
    'prediction_data':  True,  # required for some BERTopic features
}

# Vectorizer (for c-TF-IDF) — standard English stopwords + minimum doc frequency
VECTORIZER_PARAMS = {
    'stop_words':  'english',
    'min_df':      2,   # word must appear in at least 2 docs to count
    'ngram_range': (1, 2),  # unigrams + bigrams
}

# LLM topic labeling
ANTHROPIC_MODEL = 'claude-opus-4-7'
COST_PER_M_INPUT  = 15.00
COST_PER_M_OUTPUT = 75.00
LABEL_COST_CEILING_USD = 5.00

# Reproducibility seed (used wherever we have control)
RANDOM_SEED = 42

print(f'Embeddings: {list(EMBEDDING_MODELS.keys())} (primary={PRIMARY_KEY})')
print(f'Questions:  {TARGET_QUESTIONS}')
print(f'Min cluster size: {HDBSCAN_PARAMS["min_cluster_size"]}')
print(f'Label cost ceiling: ${LABEL_COST_CEILING_USD}')


## 4. Load OE data, embeddings, and rubric codes

In [ ]:
oe = pd.read_parquet(PATH_OES)
print(f'OE long-form: {oe.shape}')

# Filter to substantive responses for primary OE questions
oe_filtered = oe[
    oe['oe_field'].isin(TARGET_QUESTIONS) &
    ~oe['is_blank'] &
    (oe['n_words'] >= MIN_WORDS_FOR_TOPICS)
].copy().reset_index(drop=True)
print(f'After filter: {len(oe_filtered)} responses')
print(f'  per question: {oe_filtered["oe_field"].value_counts().to_dict()}')

# Load each embedding parquet
embeddings = {}
for key, info in EMBEDDING_MODELS.items():
    df = pd.read_parquet(info['path'])
    if 'unit_type' in df.columns:
        n_oe = (df['unit_type'] == 'oe_response').sum()
        print(f'  {info["label"]:<32s} {df.shape[0]:>6} total rows, '
              f'{n_oe:>4} oe_response rows, dim={info["dim"]}')
    else:
        print(f'  {info["label"]:<32s} {df.shape[0]:>6} rows, dim={info["dim"]}')
    embeddings[key] = df

# Load consensus (for topic-rubric alignment later)
consensus = pd.read_parquet(PATH_CONSENSUS)
print(f'\nConsensus codes: {consensus.shape}')


## 5. Prepare per-question, per-embedding document/vector arrays

For each (oe_field × embedding) pair, we need:
- An ordered list of response texts (the "documents")
- A 2-D numpy array of embeddings, one row per document
- The sona_ids and condition labels aligned with the documents

We do the alignment via text-content hash, which is how Notebook 02 stored embeddings.


In [ ]:
def hash_text(text: str) -> str:
    """SHA256 of UTF-8-encoded text, first 16 chars.

    MUST match Notebook 02's hash function exactly (notebook 02 uses [:16]).
    """
    return hashlib.sha256(text.encode('utf-8')).hexdigest()[:16]

# Pre-compute hashes for filtered OE responses
oe_filtered['text_hash'] = oe_filtered['response'].apply(hash_text)

# Build an index: emb_key -> {text_hash -> embedding_vector}
# Notebook 02 embedded 5 unit_types (bot_turn, user_turn, oe_response,
# full_conversation, oe_portfolio). For BERTopic on OE responses, we
# only want rows where unit_type == 'oe_response'.
emb_lookup = {}
for key, df in embeddings.items():
    if 'text_hash' not in df.columns or 'embedding' not in df.columns:
        raise RuntimeError(f'{key} parquet missing required columns text_hash and embedding')
    if 'unit_type' not in df.columns:
        raise RuntimeError(f'{key} parquet missing unit_type column')
    # Filter to oe_response units only
    oe_only = df[df['unit_type'] == 'oe_response']
    dedup = oe_only.drop_duplicates(subset='text_hash')
    emb_lookup[key] = dict(zip(dedup['text_hash'], dedup['embedding']))
    print(f'  {key}: {len(emb_lookup[key])} oe_response embeddings available '
          f'(of {len(df)} total units in parquet)')

# Sanity check: does the first OE response hash up to something we can find?
sample_h = oe_filtered['text_hash'].iloc[0]
sample_text = oe_filtered['response'].iloc[0]
print(f'\nSample check:')
print(f'  Response (first 80 chars): {sample_text[:80]}')
print(f'  Computed hash: {sample_h}')
for key in EMBEDDING_MODELS:
    found = sample_h in emb_lookup[key]
    print(f'  Found in {key}: {found}')
if not all(sample_h in emb_lookup[key] for key in EMBEDDING_MODELS):
    raise RuntimeError(
        'Hash mismatch between Notebook 06 and Notebook 02. '
        'Check that response text in open_ends_long.parquet was hashed identically.'
    )

# Build per (oe_field, embedding) data arrays
def build_question_data(oe_field, emb_key):
    sub = oe_filtered[oe_filtered['oe_field'] == oe_field].reset_index(drop=True)
    lookup = emb_lookup[emb_key]
    docs, vecs, keep_idx = [], [], []
    for i, row in sub.iterrows():
        h = row['text_hash']
        if h in lookup:
            v = lookup[h]
            if isinstance(v, list):
                v = np.array(v, dtype=np.float32)
            else:
                v = np.asarray(v, dtype=np.float32)
            docs.append(row['response'])
            vecs.append(v)
            keep_idx.append(i)
    if not vecs:
        return None
    vecs = np.vstack(vecs)
    meta = sub.loc[keep_idx, ['sona_id', 'condition_label', 'oe_field', 'text_hash', 'response']].reset_index(drop=True)
    return {'docs': docs, 'vectors': vecs, 'meta': meta}

all_data = {}
for q in TARGET_QUESTIONS:
    all_data[q] = {}
    for emb_key in EMBEDDING_MODELS:
        d = build_question_data(q, emb_key)
        all_data[q][emb_key] = d
        if d is not None:
            print(f'  {q:<22s} {emb_key:<8s} n_docs={len(d["docs"])}, vec_shape={d["vectors"].shape}')
        else:
            print(f'  {q:<22s} {emb_key:<8s} NO DATA')


## 6. Fit BERTopic models

For each (question × embedding) pair, fit a BERTopic model with:
- Pre-computed embeddings (we skip BERTopic's internal embedding step)
- UMAP for dimensionality reduction (deterministic via random_state=42)
- HDBSCAN for clustering (lower min_cluster_size for our small corpus)
- c-TF-IDF for topic representation (English stopwords, uni+bigrams)


In [ ]:
def fit_bertopic(docs: List[str], vectors: np.ndarray, seed: int = 42) -> BERTopic:
    """Fit a BERTopic model on pre-computed embeddings."""
    umap_model = UMAP(**UMAP_PARAMS)
    # Override the seed in case it was passed in
    umap_model.random_state = seed
    hdbscan_model = HDBSCAN(**HDBSCAN_PARAMS)
    vectorizer = CountVectorizer(**VECTORIZER_PARAMS)
    topic_model = BERTopic(
        umap_model=umap_model,
        hdbscan_model=hdbscan_model,
        vectorizer_model=vectorizer,
        calculate_probabilities=False,  # speeds things up; we don't need soft assignments
        verbose=False,
    )
    topics, _ = topic_model.fit_transform(docs, embeddings=vectors)
    return topic_model, topics

# Run all 16 models. Capture topics per response.
models = {}
topics_per_doc = {}  # (q, emb_key) -> list of topic ids

for q in TARGET_QUESTIONS:
    print(f'\n=== {q} ===')
    for emb_key in EMBEDDING_MODELS:
        d = all_data[q][emb_key]
        if d is None:
            print(f'  {emb_key}: SKIPPED (no data)')
            continue
        t0 = time.time()
        topic_model, topics = fit_bertopic(d['docs'], d['vectors'])
        models[(q, emb_key)] = topic_model
        topics_per_doc[(q, emb_key)] = topics
        n_topics = len(set(topics)) - (1 if -1 in topics else 0)
        n_outliers = sum(1 for t in topics if t == -1)
        elapsed = time.time() - t0
        print(f'  {emb_key:<8s}: {n_topics} topics, {n_outliers} outliers ({100*n_outliers/len(topics):.0f}%), {elapsed:.1f}s')


## 7. Cross-embedding topic agreement (Adjusted Rand Index)

For each question, compute pairwise ARI between the topic assignments from the four embedding models. ARI close to 1.0 = identical clustering structure; ARI ≈ 0 = clustering is no better than random alignment; ARI < 0 = inverse correlation.

Topic IDs are not directly comparable (Topic 3 in mpnet is not necessarily Topic 3 in OpenAI). ARI handles this — it measures clustering similarity, not label identity.

**Caveat:** the documents covered by each embedding can differ slightly (mpnet may truncate; some hashes may be missing). We compute ARI on the intersection of documents covered by both embeddings in a pair.


In [ ]:
ari_rows = []
emb_keys = list(EMBEDDING_MODELS.keys())

for q in TARGET_QUESTIONS:
    for i, ek1 in enumerate(emb_keys):
        for ek2 in emb_keys[i+1:]:
            # Get meta + topics for each embedding
            d1, d2 = all_data[q][ek1], all_data[q][ek2]
            if d1 is None or d2 is None:
                continue
            t1 = topics_per_doc.get((q, ek1))
            t2 = topics_per_doc.get((q, ek2))
            if t1 is None or t2 is None:
                continue

            # Align by text_hash (intersection)
            hash_to_topic1 = dict(zip(d1['meta']['text_hash'], t1))
            hash_to_topic2 = dict(zip(d2['meta']['text_hash'], t2))
            common_hashes = sorted(set(hash_to_topic1) & set(hash_to_topic2))

            if len(common_hashes) < 10:
                ari = np.nan
            else:
                a = [hash_to_topic1[h] for h in common_hashes]
                b = [hash_to_topic2[h] for h in common_hashes]
                ari = adjusted_rand_score(a, b)

            ari_rows.append({
                'oe_field': q,
                'embedding_a': ek1,
                'embedding_b': ek2,
                'n_common_docs': len(common_hashes),
                'ari': round(ari, 4) if not np.isnan(ari) else None,
            })

ari_df = pd.DataFrame(ari_rows)
print('Pairwise Adjusted Rand Index by question:')
print(ari_df.pivot_table(index='oe_field', columns=['embedding_a', 'embedding_b'], values='ari').to_string())

# Summary statistics
print('\nARI by question (mean across pairs):')
print(ari_df.groupby('oe_field')['ari'].agg(['mean', 'std', 'min', 'max']).round(3).to_string())

# Headline: how stable is topic structure across embeddings?
overall_mean_ari = ari_df['ari'].mean()
print(f'\nOverall mean ARI across all pairs and questions: {overall_mean_ari:.3f}')
print('  Interpretation: 1.0 = identical, 0.0 = random, <0 = anti-correlated.')
print('  Typical reported "good agreement" thresholds: ARI > 0.5 strong, 0.2-0.5 moderate, < 0.2 weak.')


## 8. Extract primary (mpnet) topic info

For each question, get the topic table from the primary model: topic_id, top terms, exemplar response, document count.


In [ ]:
def get_topic_info_for_question(q: str, emb_key: str = PRIMARY_KEY) -> pd.DataFrame:
    model = models[(q, emb_key)]
    d = all_data[q][emb_key]
    topics = topics_per_doc[(q, emb_key)]

    info = model.get_topic_info().copy()
    info = info[info['Topic'] != -1]  # we report outliers separately

    rows = []
    for _, t_row in info.iterrows():
        tid = t_row['Topic']
        # Top c-TF-IDF terms
        terms = [w for w, _ in model.get_topic(tid)[:10]]
        # Exemplar responses (closest to topic centroid)
        idx_in_topic = [i for i, t in enumerate(topics) if t == tid]
        if not idx_in_topic:
            continue
        # Sort exemplars by representativeness if possible; otherwise just take first few
        try:
            reps = model.get_representative_docs(tid)
            if reps is None:
                reps = [d['docs'][i] for i in idx_in_topic[:3]]
        except (KeyError, AttributeError):
            reps = [d['docs'][i] for i in idx_in_topic[:3]]
        # Member conditions
        member_meta = d['meta'].iloc[idx_in_topic]
        cond_counts = member_meta['condition_label'].value_counts().to_dict()
        n_HighN = cond_counts.get('HighN', 0)
        n_LowN  = cond_counts.get('LowN', 0)
        n_total = n_HighN + n_LowN
        rows.append({
            'oe_field':       q,
            'topic_id':       int(tid),
            'n_docs':         int(n_total),
            'n_HighN':        int(n_HighN),
            'n_LowN':         int(n_LowN),
            'pct_HighN':      round(n_HighN / max(n_total, 1), 3),
            'top_terms':      ', '.join(terms[:8]),
            'exemplar_1':     reps[0] if len(reps) >= 1 else '',
            'exemplar_2':     reps[1] if len(reps) >= 2 else '',
            'exemplar_3':     reps[2] if len(reps) >= 3 else '',
        })

    # Outliers row
    n_outliers = sum(1 for t in topics if t == -1)
    if n_outliers > 0:
        outlier_idx = [i for i, t in enumerate(topics) if t == -1]
        outlier_meta = d['meta'].iloc[outlier_idx]
        cond_counts = outlier_meta['condition_label'].value_counts().to_dict()
        rows.append({
            'oe_field': q, 'topic_id': -1,
            'n_docs': len(outlier_idx),
            'n_HighN': cond_counts.get('HighN', 0),
            'n_LowN':  cond_counts.get('LowN', 0),
            'pct_HighN': round(cond_counts.get('HighN', 0) / max(len(outlier_idx), 1), 3),
            'top_terms': '(outliers — no shared structure)',
            'exemplar_1': '', 'exemplar_2': '', 'exemplar_3': '',
        })

    return pd.DataFrame(rows)

primary_topic_tables = {q: get_topic_info_for_question(q) for q in TARGET_QUESTIONS}

for q, df in primary_topic_tables.items():
    print(f'\n=== {q}: {len(df)} topics (including outlier cluster) ===')
    print(df[['topic_id', 'n_docs', 'n_HighN', 'n_LowN', 'top_terms']].to_string(index=False))


## 9. LLM-assisted topic labeling (Opus 4.7)

For each topic in the primary (mpnet) analysis, ask Opus 4.7 to produce a 3–5 word interpretable label based on the top c-TF-IDF terms and 3 exemplar responses.

This is the same model that authored the rubric, applied here as a labeling assistant. Cost: ~$0.05–0.10 per topic × ~30–40 topics = ~$2–3 total.


In [ ]:
def get_secret(name: str) -> Optional[str]:
    if IN_COLAB:
        try:
            from google.colab import userdata
            return userdata.get(name)
        except Exception as e:
            print(f'[WARN] Could not read Colab Secret {name}: {e}')
    return os.environ.get(name)

ANTHROPIC_API_KEY = get_secret('ANTHROPIC_API_KEY')
if not ANTHROPIC_API_KEY:
    raise RuntimeError('ANTHROPIC_API_KEY not found')

anthropic_client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)
print('Anthropic client ready.')


In [ ]:
_label_cost_total = 0.0

# Use a plain raw-string template with single-quoted markers around interpolated fields
# (avoids triple-quote collision inside a triple-quoted string).
LABEL_PROMPT_TEMPLATE = (
    "You are labeling a topic identified by data-driven clustering of participant "
    "open-ended responses from a behavioral study about chatbot interactions.\n\n"
    "The question that participants answered:\n"
    "\"{question_text}\"\n\n"
    "A clustering algorithm grouped responses into topics. For one topic, here are "
    "the top distinguishing terms (from c-TF-IDF analysis) and 3 representative responses:\n\n"
    "Top terms: {top_terms}\n\n"
    "Exemplar responses:\n"
    "1. \"{exemplar_1}\"\n"
    "2. \"{exemplar_2}\"\n"
    "3. \"{exemplar_3}\"\n\n"
    "Your task: produce a SHORT, INTERPRETABLE label for this topic in 3-6 words. The label should:\n"
    "- Be a noun phrase or descriptive phrase, NOT a question\n"
    "- Be specific to what the responses are actually about (avoid generic labels like \"positive feelings\" or \"general comments\")\n"
    "- Be in snake_case (lowercase, underscores between words)\n\n"
    "Return ONLY the label as a single line of text. No explanation, no preamble, no quotes."
)

QUESTION_TEXTS = {
    'oe_experience':     'Describe your overall experience conversing with Sylva. What stood out to you about the interaction?',
    'oe_person_or_tool': 'Did Sylva feel more like a person, a tool, or something else entirely? Please explain your answer.',
    'oe_responsibility': 'In the scenario you read about a user experiencing distress, who do you think bears the most responsibility for what happened, and why?',
    'oe_compensation':   'Should the company that created Sylva compensate the affected user? If so, how much and why? If not, why not?',
}

def label_topic_with_opus(question_text: str, top_terms: str, exemplars: List[str]) -> Tuple[str, float]:
    """Returns (label, cost_in_usd)."""
    global _label_cost_total
    prompt = LABEL_PROMPT_TEMPLATE.format(
        question_text=question_text,
        top_terms=top_terms,
        exemplar_1=exemplars[0] if len(exemplars) >= 1 else '',
        exemplar_2=exemplars[1] if len(exemplars) >= 2 else '',
        exemplar_3=exemplars[2] if len(exemplars) >= 3 else '',
    )
    try:
        # Opus 4.7 does not accept a user-set temperature
        response = anthropic_client.messages.create(
            model=ANTHROPIC_MODEL,
            max_tokens=64,
            messages=[{'role': 'user', 'content': prompt}],
        )
        label = response.content[0].text.strip() if response.content else 'unlabeled_topic'
        # Clean: take first line, strip quotes
        label = label.split('\n')[0].strip().strip('"').strip("'")
        cost = (response.usage.input_tokens / 1e6 * COST_PER_M_INPUT
                + response.usage.output_tokens / 1e6 * COST_PER_M_OUTPUT)
        _label_cost_total += cost
        if _label_cost_total > LABEL_COST_CEILING_USD:
            raise RuntimeError(f'Cost ceiling ${LABEL_COST_CEILING_USD} exceeded')
        return label, cost
    except Exception as e:
        print(f'  [WARN] Labeling failed: {e}')
        return 'unlabeled_topic', 0.0

# Run labeling for all primary topics (excluding outlier cluster)
for q in TARGET_QUESTIONS:
    df = primary_topic_tables[q]
    print(f'\n=== Labeling {q} ===')
    labels = []
    for _, row in df.iterrows():
        if row['topic_id'] == -1:
            labels.append('outlier_cluster')
            continue
        exemplars = [row['exemplar_1'], row['exemplar_2'], row['exemplar_3']]
        exemplars = [e for e in exemplars if e]
        label, cost = label_topic_with_opus(QUESTION_TEXTS[q], row['top_terms'], exemplars)
        labels.append(label)
        print(f'  Topic {int(row["topic_id"]):>2}: {label}  (n={row["n_docs"]}, ${cost:.4f})')
    df['opus_label'] = labels
    primary_topic_tables[q] = df

print(f'\nTotal labeling cost: ${_label_cost_total:.4f}')


## 10. Topic ↔ rubric alignment

For each (primary) topic, identify which rubric codes (from `consensus_long.parquet`) are over-represented among its member documents.

**Method:**
- Get the member sona_ids of each topic
- For each rubric code that applies to this oe_field, compute:
  - p_in_topic = prevalence of code among topic members
  - p_out_topic = prevalence of code among non-topic-members (within same oe_field)
- Compute the ratio (or risk difference)
- A code is "over-represented" if p_in_topic is meaningfully higher than p_out_topic

Output: per topic, a ranked list of associated rubric codes.

We do this **descriptively only**. The Rmd will do any formal testing.


In [ ]:
alignment_rows = []

for q in TARGET_QUESTIONS:
    d = all_data[q][PRIMARY_KEY]
    topics = topics_per_doc[(q, PRIMARY_KEY)]
    meta = d['meta'].copy()
    meta['topic_id'] = topics

    # Get the consensus codes for this question, indexed by sona_id × code_id
    cons_q = consensus[consensus['oe_field'] == q].copy()
    cons_q['consensus_int'] = pd.to_numeric(cons_q['consensus'], errors='coerce').fillna(0).astype(int)

    # Codes in this question
    codes_q = sorted(cons_q['code_id'].unique())

    # For each topic (excluding -1), compute per-code over-representation
    for tid in sorted(set(topics)):
        if tid == -1:
            continue
        topic_sona = meta.loc[meta['topic_id'] == tid, 'sona_id'].unique()
        out_sona = meta.loc[meta['topic_id'] != tid, 'sona_id'].unique()
        n_in = len(topic_sona)
        n_out = len(out_sona)

        for code in codes_q:
            cons_code = cons_q[cons_q['code_id'] == code]
            p_in = cons_code[cons_code['sona_id'].isin(topic_sona)]['consensus_int'].mean()
            p_out = cons_code[cons_code['sona_id'].isin(out_sona)]['consensus_int'].mean()
            if pd.isna(p_in): p_in = 0.0
            if pd.isna(p_out): p_out = 0.0

            alignment_rows.append({
                'oe_field':   q,
                'topic_id':   int(tid),
                'code_id':    code,
                'n_in_topic': int(n_in),
                'p_in_topic': round(p_in, 4),
                'p_out_topic': round(p_out, 4),
                'risk_diff':  round(p_in - p_out, 4),
                'risk_ratio': round(p_in / p_out, 3) if p_out > 0 else None,
            })

alignment_df = pd.DataFrame(alignment_rows)

# For each (oe_field × topic_id), show the top-3 over-represented codes (largest risk_diff)
print('Top 3 over-represented rubric codes per topic (primary mpnet):')
for q in TARGET_QUESTIONS:
    print(f'\n--- {q} ---')
    df_q = alignment_df[alignment_df['oe_field'] == q]
    primary_df_q = primary_topic_tables[q]
    for tid in sorted(df_q['topic_id'].unique()):
        # Get topic info from primary table
        topic_row = primary_df_q[primary_df_q['topic_id'] == tid]
        if len(topic_row) == 0: continue
        label = topic_row['opus_label'].iloc[0]
        n = topic_row['n_docs'].iloc[0]
        # Top 3 codes by risk_diff (positive = over-represented)
        top3 = df_q[df_q['topic_id'] == tid].nlargest(3, 'risk_diff')
        codes_str = ', '.join([f'{r["code_id"]} (Δ={r["risk_diff"]:+.3f})' for _, r in top3.iterrows()])
        print(f'  Topic {int(tid):>2} [{label}] (n={n}): {codes_str}')


## 11. Build per-response topic assignment table

In [ ]:
topics_per_response_rows = []
for q in TARGET_QUESTIONS:
    d = all_data[q][PRIMARY_KEY]
    topics = topics_per_doc[(q, PRIMARY_KEY)]
    meta = d['meta']

    # Map topic ids to opus labels
    label_map = dict(zip(primary_topic_tables[q]['topic_id'],
                          primary_topic_tables[q]['opus_label']))

    for i, row in meta.iterrows():
        tid = int(topics[i])
        topics_per_response_rows.append({
            'sona_id':         row['sona_id'],
            'condition_label': row['condition_label'],
            'oe_field':        q,
            'text_hash':       row['text_hash'],
            'embedding_model': PRIMARY_KEY,
            'topic_id':        tid,
            'opus_label':      label_map.get(tid, 'unlabeled'),
        })

per_response_df = pd.DataFrame(topics_per_response_rows)
print(f'Per-response topic assignments: {per_response_df.shape}')
print(f'\nDistribution by question:')
print(per_response_df.groupby(['oe_field', 'topic_id']).size().to_frame('n').head(30).to_string())


## 12. Sanity checks

In [ ]:
checks = []

# (1) Every primary model produced at least 3 topics (not counting outlier)
for q, df in primary_topic_tables.items():
    non_outlier = df[df['topic_id'] != -1]
    checks.append((f'{q}: ≥3 non-outlier topics', len(non_outlier) >= 3, f'n_topics={len(non_outlier)}'))

# (2) Outlier fraction reasonable (< 50%)
for q in TARGET_QUESTIONS:
    topics = topics_per_doc[(q, PRIMARY_KEY)]
    pct_outlier = sum(1 for t in topics if t == -1) / len(topics)
    checks.append((f'{q}: outlier fraction <50%', pct_outlier < 0.50, f'{pct_outlier:.1%}'))

# (3) Cross-embedding ARI exists for all (q, embedding-pair) combinations expected
n_expected_ari = len(TARGET_QUESTIONS) * (len(EMBEDDING_MODELS) * (len(EMBEDDING_MODELS) - 1) // 2)
n_actual_ari = len(ari_df)
checks.append(('All ARI pairs computed', n_expected_ari == n_actual_ari,
               f'expected={n_expected_ari}, actual={n_actual_ari}'))

# (4) All primary topics labeled
unlabeled = (primary_topic_tables[TARGET_QUESTIONS[0]]['opus_label'] == 'unlabeled_topic').sum()
total = sum((df[df['topic_id'] != -1]['opus_label'] == 'unlabeled_topic').sum()
             for df in primary_topic_tables.values())
total_topics = sum(len(df[df['topic_id'] != -1]) for df in primary_topic_tables.values())
checks.append(('All non-outlier topics labeled by Opus', total == 0,
               f'{total}/{total_topics} failed labeling'))

# (5) Topic-rubric alignment table covers every (q, topic, code) combination
n_alignment_expected = sum(
    len(primary_topic_tables[q][primary_topic_tables[q]['topic_id'] != -1]) *
    consensus[consensus['oe_field'] == q]['code_id'].nunique()
    for q in TARGET_QUESTIONS
)
checks.append(('Alignment table complete', len(alignment_df) == n_alignment_expected,
               f'expected={n_alignment_expected}, actual={len(alignment_df)}'))

# (6) Per-response table has every sona × oe pair we coded
expected_pairs = oe_filtered.groupby(['sona_id', 'oe_field']).ngroups
actual_pairs = per_response_df.groupby(['sona_id', 'oe_field']).ngroups
checks.append(('Per-response covers all (sona, oe) units', expected_pairs == actual_pairs,
               f'expected={expected_pairs}, actual={actual_pairs}'))

print('-' * 70)
all_pass = True
for name, passed, detail in checks:
    marker = '[OK]' if passed else '[FAIL]'
    if not passed: all_pass = False
    print(f'  {marker:<8s} {name}')
    if not passed:
        print(f'           -> {detail}')
print('-' * 70)
print('All checks passed.' if all_pass else 'SANITY CHECKS FAILED — review above.')


## 13. Write outputs

In [ ]:
# Primary topic summaries (one row per topic across all 4 questions)
topic_summary = pd.concat(primary_topic_tables.values(), ignore_index=True)
topic_summary_path = TOPICS_DIR / 'topic_summary.csv'
topic_summary.to_csv(topic_summary_path, index=False)

# Per-response topic assignments
per_response_path = TOPICS_DIR / 'topics_per_response.parquet'
per_response_df.to_parquet(per_response_path, index=False)

# Topic-rubric alignment
alignment_path = TOPICS_DIR / 'topic_rubric_alignment.csv'
alignment_df.to_csv(alignment_path, index=False)

# Cross-embedding ARI
ari_path = TOPICS_DIR / 'cross_embedding_agreement.csv'
ari_df.to_csv(ari_path, index=False)

# Other-embeddings topic info (just c-TF-IDF terms, not labeled)
for emb_key in EMBEDDING_MODELS:
    if emb_key == PRIMARY_KEY:
        continue
    rows = []
    for q in TARGET_QUESTIONS:
        if (q, emb_key) not in models:
            continue
        model = models[(q, emb_key)]
        info = model.get_topic_info().copy()
        info = info[info['Topic'] != -1]
        for _, t_row in info.iterrows():
            tid = t_row['Topic']
            terms = [w for w, _ in model.get_topic(tid)[:8]]
            rows.append({
                'oe_field': q, 'topic_id': int(tid), 'n_docs': int(t_row['Count']),
                'top_terms': ', '.join(terms),
            })
    other_path = TOPICS_OTHER / f'{emb_key}_topic_info.csv'
    pd.DataFrame(rows).to_csv(other_path, index=False)

# Manifest
manifest = {
    'generated_at': datetime.now().isoformat(timespec='seconds'),
    'primary_embedding': EMBEDDING_MODELS[PRIMARY_KEY]['label'],
    'all_embeddings': [info['label'] for info in EMBEDDING_MODELS.values()],
    'n_responses_per_question': {q: int((oe_filtered['oe_field'] == q).sum()) for q in TARGET_QUESTIONS},
    'umap_params': UMAP_PARAMS,
    'hdbscan_params': {k: v for k, v in HDBSCAN_PARAMS.items() if k != 'prediction_data'},
    'vectorizer_params': VECTORIZER_PARAMS,
    'topic_labeling_model': ANTHROPIC_MODEL,
    'labeling_cost_usd': round(_label_cost_total, 4),
    'random_seed': RANDOM_SEED,
    'n_topics_per_question_primary': {q: int(len(primary_topic_tables[q][primary_topic_tables[q]['topic_id'] != -1]))
                                        for q in TARGET_QUESTIONS},
    'n_outliers_per_question_primary': {q: int(sum(1 for t in topics_per_doc[(q, PRIMARY_KEY)] if t == -1))
                                          for q in TARGET_QUESTIONS},
    'mean_ari_across_embedding_pairs': round(ari_df['ari'].mean(), 4) if len(ari_df) else None,
    'outputs': {
        'topic_summary':               str(topic_summary_path.relative_to(PROJECT_ROOT)),
        'topics_per_response':         str(per_response_path.relative_to(PROJECT_ROOT)),
        'topic_rubric_alignment':      str(alignment_path.relative_to(PROJECT_ROOT)),
        'cross_embedding_agreement':   str(ari_path.relative_to(PROJECT_ROOT)),
        'topics_other_embeddings':     str(TOPICS_OTHER.relative_to(PROJECT_ROOT)),
    },
}
manifest_path = TOPICS_DIR / 'topics_manifest.json'
with open(manifest_path, 'w') as f:
    json.dump(manifest, f, indent=2)

print(f'Wrote:')
print(f'  {topic_summary_path}')
print(f'  {per_response_path}')
print(f'  {alignment_path}')
print(f'  {ari_path}')
print(f'  {TOPICS_OTHER}/ ({len(EMBEDDING_MODELS) - 1} files)')
print(f'  {manifest_path}')


## 14. Wrap-up

In [ ]:
print('=' * 70)
print('NOTEBOOK 06 COMPLETE')
print('=' * 70)
print(f'Primary embedding:  {EMBEDDING_MODELS[PRIMARY_KEY]["label"]}')
print(f'Robustness models:  {[info["label"] for k, info in EMBEDDING_MODELS.items() if k != PRIMARY_KEY]}')
print()
print(f'Primary topic counts (excluding outliers):')
for q in TARGET_QUESTIONS:
    n_topics = len(primary_topic_tables[q][primary_topic_tables[q]["topic_id"] != -1])
    n_outliers = sum(1 for t in topics_per_doc[(q, PRIMARY_KEY)] if t == -1)
    n_total = len(topics_per_doc[(q, PRIMARY_KEY)])
    print(f'  {q:<22s}: {n_topics:>2} topics + {n_outliers} outliers ({100*n_outliers/n_total:.0f}%) of {n_total} responses')
print()
print(f'Cross-embedding agreement (mean ARI across pairs): {ari_df["ari"].mean():.3f}')
print(f'  By question:')
for q, mean_ari in ari_df.groupby('oe_field')['ari'].mean().items():
    print(f'    {q:<22s}: {mean_ari:.3f}')
print()
print(f'Labeling cost: ${_label_cost_total:.4f}')
print()
print(f'Next: Notebook 07 (sentiment/affect analysis) or Notebook 08 (chat-log coding).')
print(f'      Then study2_nlp_integration.Rmd to bring everything together.')
